# Colab TTS Quality Gate

Tests REAL Qwen3-TTS and Chatterbox-Turbo on a Colab GPU.


In [ ]:
import sys, torch
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torch CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not available. In Colab select Runtime > Change runtime type > GPU, then restart and Run all.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# Install TTS packages only. IMPORTANT: do not reinstall torch in Colab.
# This avoids the Cell 2 CUDA/PyTorch conflict.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'qwen-tts', 'chatterbox-tts', 'soundfile', 'scipy', 'numpy'])
print('TTS packages installed.')
print('Restart is NOT required unless pip reports a torch dependency conflict.')

In [ ]:
from pathlib import Path
import json, subprocess, time, traceback
import torch
out_dir=Path('/content/openmontage-colab/projects/colab-tts-quality'); out_dir.mkdir(parents=True,exist_ok=True)
qwen_path=out_dir/'qwen3_tts_output.wav'; chatter_path=out_dir/'chatterbox_output.wav'; report_path=out_dir/'tts_quality_report.json'
text='A hundred years ago, humanity looked toward the stars and wondered whether we were alone. Tonight, something answered. The signal came from a world no telescope had ever seen before. And buried inside that transmission was a message meant for us.'
instruction='Calm cinematic documentary narration. Natural pacing, clear pronunciation, subtle mystery and anticipation, and natural emotional expression.'
def valid_wav(path):
    if not path.exists() or path.stat().st_size < 1000: return False
    p=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],capture_output=True,text=True)
    try: return p.returncode==0 and float(p.stdout.strip())>0.1
    except: return False

In [ ]:
qwen={'status':'NOT_RUN','model':'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice'}
try:
    from qwen_tts import Qwen3TTS
    import soundfile as sf, numpy as np
    t=time.time()
    model=Qwen3TTS('Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice', device_map='cuda:0', dtype=torch.bfloat16)
    qwen['load_seconds']=round(time.time()-t,2)
    t=time.time()
    result=model.generate_custom_voice(text=text, language='English', speaker='Ryan', instruct=instruction)
    wav,sr=result[0]
    sf.write(str(qwen_path),np.asarray(wav),int(sr))
    qwen.update({'generation_seconds':round(time.time()-t,2),'sample_rate':int(sr),'status':'REAL_PASS' if valid_wav(qwen_path) else 'REAL_FAIL'})
except Exception as e:
    qwen.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})
print('Qwen:',qwen['status'])

In [ ]:
chatter={'status':'NOT_RUN','model':'ResembleAI/chatterbox-turbo'}
try:
    from chatterbox.tts_turbo import ChatterboxTurboTTS
    import torchaudio
    t=time.time()
    model2=ChatterboxTurboTTS.from_pretrained(device='cuda')
    chatter['load_seconds']=round(time.time()-t,2)
    t=time.time(); wav=model2.generate(text)
    torchaudio.save(str(chatter_path),wav.cpu(),model2.sr)
    chatter.update({'generation_seconds':round(time.time()-t,2),'sample_rate':int(model2.sr),'status':'REAL_PASS' if valid_wav(chatter_path) else 'REAL_FAIL'})
except Exception as e:
    chatter.update({'status':'REAL_FAIL','error':str(e),'traceback':traceback.format_exc()})
print('Chatterbox:',chatter['status'])

In [ ]:
from IPython.display import Audio, display
report={'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'cuda':torch.version.cuda,'qwen':qwen,'chatterbox':chatter}
report_path.write_text(json.dumps(report,indent=2,default=str),encoding='utf-8')
print(json.dumps(report,indent=2,default=str))
if qwen['status']=='REAL_PASS': display(Audio(str(qwen_path)))
if chatter['status']=='REAL_PASS': display(Audio(str(chatter_path)))
print('Report:',report_path)